# CQL vs Expert policy comparison

This notebook loads the per-query comparison CSV and the raw trajectory CSVs for both policies, builds a single pandas dataset, and visualizes the most important statistics.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path("..")
EXPERT_CSV = ROOT / "outputs" / "trajectories_expert_trec_dl_combined_50.csv"
CQL_CSV = ROOT / "outputs" / "cql_test_results_trec_dl_combined_50.csv"
PER_QUERY_CSV = ROOT / "outputs" / "cql_vs_expert_trec_dl_combined_per_query.csv"

expert = pd.read_csv(EXPERT_CSV)
cql = pd.read_csv(CQL_CSV)
per_query = pd.read_csv(PER_QUERY_CSV)

numeric_cols = ["reward", "ndcg_before", "ndcg_after", "cost", "latency_ms", "cumulative_cost"]
for col in numeric_cols:
    if col in expert.columns:
        expert[col] = pd.to_numeric(expert[col], errors="coerce")
    if col in cql.columns:
        cql[col] = pd.to_numeric(cql[col], errors="coerce")

print(f"Expert rows: {len(expert):,}  |  CQL rows: {len(cql):,}  |  Per-query rows: {len(per_query):,}")

In [ ]:
# Build per-query aggregates from raw trajectory files (cost and reward only)
# NDCG gain already comes from the pre-computed per-query CSV.
expert_agg = expert.groupby("query_id", as_index=False).agg(
    expert_total_reward=("reward", "sum"),
    expert_total_cost=("cost", "sum"),
)
cql_agg = cql.groupby("query_id", as_index=False).agg(
    cql_total_reward=("reward", "sum"),
    cql_total_cost=("cost", "sum"),
)

df = per_query.merge(expert_agg, on="query_id", how="left").merge(cql_agg, on="query_id", how="left")

df["reward_diff"] = df["cql_reward"] - df["expert_reward"]
df["cost_diff"] = df["cql_total_cost"] - df["expert_total_cost"]
df["ndcg_gain_diff"] = df["cql_ndcg_gain"] - df["expert_ndcg_gain"]

print(f"Matched queries: {len(df)}")
df.head()

## Action distribution by policy

In [ ]:
expert_actions = expert["action_name"].value_counts().reset_index()
expert_actions.columns = ["action_name", "count"]
expert_actions["policy"] = "Expert"

cql_actions = cql["action_name"].value_counts().reset_index()
cql_actions.columns = ["action_name", "count"]
cql_actions["policy"] = "CQL"

actions_df = pd.concat([expert_actions, cql_actions], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=actions_df, x="action_name", y="count", hue="policy", ax=ax)
ax.set_title("Action distribution by policy")
ax.set_xlabel("Action")
ax.set_ylabel("Number of invocations")
ax.tick_params(axis="x", rotation=30)
ax.legend(title="Policy", loc="upper right")
plt.tight_layout()
plt.show()

actions_df.pivot(index="action_name", columns="policy", values="count").fillna(0).astype(int)

## Total cost per query by policy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

x = np.arange(len(df))
width = 0.35

ax.bar(x - width / 2, df["expert_total_cost"], width, label="Expert", color="#e76f51", alpha=0.85)
ax.bar(x + width / 2, df["cql_total_cost"], width, label="CQL", color="#2a9d8f", alpha=0.85)

ax.set_xlabel("Query (index)")
ax.set_ylabel("Total cost")
ax.set_title("Total cost per query by policy")
ax.legend(title="Policy")
plt.tight_layout()
plt.show()

cost_summary = pd.DataFrame(
    {
        "Expert": [df["expert_total_cost"].sum(), df["expert_total_cost"].mean()],
        "CQL": [df["cql_total_cost"].sum(), df["cql_total_cost"].mean()],
    },
    index=["Total cost", "Mean cost per query"],
)
cost_summary

## NDCG@50 improvement per query by policy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

x = np.arange(len(df))
width = 0.35

ax.bar(x - width / 2, df["expert_ndcg_gain"], width, label="Expert", color="#e76f51", alpha=0.85)
ax.bar(x + width / 2, df["cql_ndcg_gain"], width, label="CQL", color="#2a9d8f", alpha=0.85)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Query (index)")
ax.set_ylabel("NDCG@50 improvement")
ax.set_title("NDCG@50 improvement per query by policy")
ax.legend(title="Policy")
plt.tight_layout()
plt.show()

ndcg_summary = pd.DataFrame(
    {
        "Expert": [df["expert_ndcg_gain"].mean(), df["expert_final_ndcg"].mean()],
        "CQL": [df["cql_ndcg_gain"].mean(), df["cql_final_ndcg"].mean()],
    },
    index=["Mean NDCG gain", "Mean final NDCG"],
)
ndcg_summary

## Total reward per query by policy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

x = np.arange(len(df))
width = 0.35

ax.bar(x - width / 2, df["expert_reward"], width, label="Expert", color="#e76f51", alpha=0.85)
ax.bar(x + width / 2, df["cql_reward"], width, label="CQL", color="#2a9d8f", alpha=0.85)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Query (index)")
ax.set_ylabel("Total reward")
ax.set_title("Total reward per query by policy")
ax.legend(title="Policy")
plt.tight_layout()
plt.show()

reward_summary = pd.DataFrame(
    {
        "Expert": [df["expert_reward"].mean(), df["expert_reward"].sum()],
        "CQL": [df["cql_reward"].mean(), df["cql_reward"].sum()],
    },
    index=["Mean reward per query", "Total reward"],
)
reward_summary["Difference (CQL - Expert)"] = reward_summary["CQL"] - reward_summary["Expert"]
reward_summary

## Paired reward comparison: CQL vs Expert

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

wins = df["cql_reward"] > df["expert_reward"]
colors = np.where(wins, "#2a9d8f", "#e76f51")

ax.scatter(df["expert_reward"], df["cql_reward"], c=colors, s=80, alpha=0.75, edgecolor="white", linewidth=0.6)

lo = min(df["expert_reward"].min(), df["cql_reward"].min())
hi = max(df["expert_reward"].max(), df["cql_reward"].max())
pad = max((hi - lo) * 0.05, 0.01)
ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1.2, label="Equal reward")
ax.set_xlim(lo - pad, hi + pad)
ax.set_ylim(lo - pad, hi + pad)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Expert total reward")
ax.set_ylabel("CQL total reward")
ax.set_title("Total reward per query: CQL vs Expert")
ax.legend(loc="lower right")

win_rate = wins.mean()
mean_diff = df["reward_diff"].mean()
ax.text(
    0.03,
    0.97,
    f"Matched: {len(df)}\nCQL win rate: {win_rate:.1%}\nMean diff: {mean_diff:.4f}",
    transform=ax.transAxes,
    va="top",
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.90},
)

plt.tight_layout()
plt.show()

In [ ]:
output_path = ROOT / "outputs" / "cql_vs_expert_trec_dl_comparison_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"Saved cleaned comparison dataset: {output_path}")